<a href="https://colab.research.google.com/github/sreejarepo/LLM_project1/blob/main/LLM_project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install -q transformers peft bitsandbytes datasets pypdf langdetect gdown trl accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 49.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 19.7 MB/s eta 0:00:00


In [2]:
import os
import re
import hashlib
from pypdf import PdfReader
from langdetect import detect, DetectorFactory

# Set seed for reproducible language detection
DetectorFactory.seed = 42

# =====================================================================
# STEP 1 — DOMAIN DATA COLLECTION & CLEANING
# =====================================================================

FOLDER_ID = '14Q_VaCEmqmQNmLNJlBOBzc6iHeylVoWU'
FOLDER_URL = f'https://drive.google.com/drive/folders/{FOLDER_ID}'

raw_pdf_dir = 'raw_pdfs/'
os.makedirs(raw_pdf_dir, exist_ok=True)

print("--- Step 1.1: Aggressively Downloading Compliance Folders via CLI ---")
# Using the exclamation mark CLI bypass handles nested subfolder routing reliably
!gdown --folder "{FOLDER_URL}" -O {raw_pdf_dir} --remaining-ok

# Debugging check: Confirm files exist on disk
all_downloaded_items = []
for root, dirs, files in os.walk(raw_pdf_dir):
    for file in files:
        all_downloaded_items.append(os.path.join(root, file))
print(f"\nVerification: Found {len(all_downloaded_items)} files inside the temporary directory.")



--- Step 1.1: Aggressively Downloading Compliance Folders via CLI ---
Retrieving folder contents
Retrieving folder 1Kbkvor6flZtCJ_Lpc9vIF_kalAsRjKx4 RBI
Processing file 13NTcQ8cOJkhfQ9nqpwIa_xBCrPfUv4Gy 02MD787706E669D641C090B415DBE22DEE29.PDF
Processing file 18fRJ1wcy9-9A_eArDOooDonNN5A0zlE9 03MD945692290C104C5595AEDAC89AE78788.PDF
Processing file 16hrooxCoDuHSkaZki-YVgM0Y8tNDTp8a 04MD5127742FD0914D54B5EC4ECA8076F325.PDF
Processing file 1gL9C2UiXvjG0OsLv1wrq5dyRQA5TFP2e 05MD56DF4232873F466191DC23F39F1012DD.PDF
Processing file 1jWAqQqCK0WbiR9wSrmdIE7rH091G9ZqE 06MDE170516F633150EBCFE438084174F7DECCDC20C.PDF
Processing file 10F2aEIgCHCv_KVbiqauvo1uiF4oLQntR 07MIP0101168C51C4CB845342D09E8A4A613B1F1C6D.PDF
Processing file 14lnQmf3nY4akR0bld3yrCJG6EZWYK4Nt 08MDROA112016989330DF1FF9494784DAC32B488E6AE4.PDF
Processing file 1w2X6fqJLua0GtUhlTpiJuDNUCGXFdGpp Accessibility20012026.pdf
Processing file 1sxvtBmoe_eU9cfuPKygHZ9m5z-DzQryl Utkarsh202910042026.pdf
Processing file 1080UwsqBi-OYTBJm2t0_

In [3]:
# Setup Output Structure
output_corpus_dir = 'domain_corpus/'
os.makedirs(output_corpus_dir, exist_ok=True)

MIN_CHAR_LENGTH = 200  # Threshold justification: Removes headers/blank placeholder items
seen_page_hashes = set()

stats = {
    "raw_pdfs": 0,
    "total_pages_read": 0,
    "filtered_short": 0,
    "filtered_duplicate": 0,
    "filtered_non_english": 0,
    "extracted_txt_files": 0
}

# Recursively locate every .pdf across downloaded directories
pdf_paths = []
for root, dirs, files in os.walk(raw_pdf_dir):
    for file in files:
        if file.endswith('.pdf'):
            pdf_paths.append(os.path.join(root, file))

stats["raw_pdfs"] = len(pdf_paths)
print(f"\n--- Step 1.2: Processing {len(pdf_paths)} Cleaned PDF Files Recursively ---")

for pdf_path in pdf_paths:
    # Safe regex string cleaning using tuple slicing [0] to avoid string casting errors
    base_name = os.path.splitext(os.path.basename(pdf_path))[0]
    clean_name = re.sub(r'[^a-zA-Z0-9_\-]', '_', base_name)
    print("base_name",base_name)
    txt_filename = f"{clean_name}.txt"
    txt_path = os.path.join(output_corpus_dir, txt_filename)

    cleaned_text_content = []

    try:
        reader = PdfReader(pdf_path)
        print("reading pdf")
        count=0
        for page in reader.pages:
            count=count+1
            stats["total_pages_read"] += 1
            text = page.extract_text() or ""
            text_striped = text.strip()

            # Filter 1: Length Threshold Filter
            if len(text_striped) < MIN_CHAR_LENGTH:
                stats["filtered_short"] += 1
                continue

            # Filter 2: Content Deduplication Hashing
            normalized_text = re.sub(r'\s+', ' ', text_striped).lower()
            page_hash = hashlib.md5(normalized_text.encode('utf-8')).hexdigest()
            if page_hash in seen_page_hashes:
                stats["filtered_duplicate"] += 1
                continue
            seen_page_hashes.add(page_hash)

            # Filter 3: English Isolation Check
            try:
                if detect(text_striped) != 'en':
                    stats["filtered_non_english"] += 1
                    continue
            except:
                stats["filtered_non_english"] += 1
                continue

            cleaned_text_content.append(text_striped)
            #print("cleaned_text")
            if count >50:
              break

        if cleaned_text_content:
            with open(txt_path, 'w', encoding='utf-8') as f:
                f.write("\n\n--- PAGE BREAK ---\n\n".join(cleaned_text_content))
            stats["extracted_txt_files"] += 1

    except Exception as e:
        print(f"Skipping unreadable file {os.path.basename(pdf_path)}: {e}")

# Render Final Report
print("\n=== PART A: PIPELINE CORPUS STATISTICS ===")
print(f"Initial PDF files identified: {stats['raw_pdfs']}")
print(f"Total processed pages: {stats['total_pages_read']}")
print(f"Pages dropped via Length Filter (< {MIN_CHAR_LENGTH} chars): {stats['filtered_short']}")
print(f"Pages dropped via Content Deduplication Hashing: {stats['filtered_duplicate']}")
print(f"Pages dropped via Non-English Language Detection: {stats['filtered_non_english']}")
print(f"Final clean text logs exported to '{output_corpus_dir}': {stats['extracted_txt_files']} files")



--- Step 1.2: Processing 3 Cleaned PDF Files Recursively ---
base_name basel-framework-full
reading pdf
base_name Accessibility20012026
reading pdf
base_name Utkarsh202910042026
reading pdf

=== PART A: PIPELINE CORPUS STATISTICS ===
Initial PDF files identified: 3
Total processed pages: 62
Pages dropped via Length Filter (< 200 chars): 5
Pages dropped via Content Deduplication Hashing: 0
Pages dropped via Non-English Language Detection: 0
Final clean text logs exported to 'domain_corpus/': 3 files


In [4]:

# =====================================================================
# STEP 2 — BASELINE MODEL OUTPUT & ARCHITECTURE REPORT
# =====================================================================
print("\n--- Step 2: Extracting Pre-Adaptation Model Architecture Baseline ---")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "openai-community/gpt2-medium"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

total_params = sum(p.numel() for p in base_model.parameters())
num_layers = len(base_model.transformer.h)
hidden_size = base_model.config.n_embd
vocab_size = base_model.config.vocab_size

print("\n=== GPT-2-MEDIUM BASELINE ARCHITECTURE REPORT ===")
print(f"Total Parameter Count: {total_params:,}")
print(f"Number of Decoder Layers: {num_layers}")
print(f"Hidden Dim Size: {hidden_size}")
print(f"Vocabulary Dimension Size: {vocab_size}")

domain_prompts = [
    "What is the maximum LTV ratio for gold loans as per RBI guidelines?",
    "What are the KYC requirements for a small savings account under RBI norms?",
    "What does SEBI LODR Regulation 17 require for board composition?"
]

print("\n=== RUNNING PRE-ADAPTATION BASELINE INFERENCES ===")
for idx, prompt in enumerate(domain_prompts, 1):
    inputs = tokenizer(prompt, return_tensors="pt").to(base_model.device)
    with torch.no_grad():
        output_tokens = base_model.generate(**inputs, max_new_tokens=100, do_sample=False)
    generated_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    print(f"\n[Prompt {idx}]: {prompt}\n[Baseline Response]: {generated_text}\n" + "-"*50)



--- Step 2: Extracting Pre-Adaptation Model Architecture Baseline ---


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.52GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


=== GPT-2-MEDIUM BASELINE ARCHITECTURE REPORT ===
Total Parameter Count: 354,823,168
Number of Decoder Layers: 24
Hidden Dim Size: 1024
Vocabulary Dimension Size: 50257

=== RUNNING PRE-ADAPTATION BASELINE INFERENCES ===

[Prompt 1]: What is the maximum LTV ratio for gold loans as per RBI guidelines?
[Baseline Response]: What is the maximum LTV ratio for gold loans as per RBI guidelines?

The maximum LTV ratio for gold loans is Rs. 1,000 per kg.

What is the maximum LTV ratio for silver loans as per RBI guidelines?

The maximum LTV ratio for silver loans is Rs. 1,000 per kg.

What is the maximum LTV ratio for platinum loans as per RBI guidelines?

The maximum LTV ratio for platinum loans is Rs. 1,000 per kg.

What is the maximum LTV ratio
--------------------------------------------------

[Prompt 2]: What are the KYC requirements for a small savings account under RBI norms?
[Baseline Response]: What are the KYC requirements for a small savings account under RBI norms?

A small saving

In [5]:
import json
import random

# For demonstration, we construct/generate pairs based on our domain context (RBI/SEBI rules)
# In production, replace this with your LLM-synthetic generation step reading from `domain_corpus/`
sample_dataset = [
    {
        "instruction": "What is the maximum LTV ratio for gold loans as per RBI guidelines?",
        "response": "Per RBI Master Direction on Loans and Advances, LTV for gold loans must not exceed 75% of the value of gold ornaments accepted as collateral."
    },
    {
        "instruction": "What are the KYC requirements for a small savings account under RBI norms?",
        "response": "Under RBI Master Direction on KYC, a small account requires a self-certified photograph and one officially valid document. Maximum balance cannot exceed Rs. 50,000."
    },
    {
        "instruction": "What does SEBI LODR Regulation 17 require for board composition?",
        "response": "SEBI LODR Regulation 17 mandates at least 50% independent directors on the board of listed entities if the chairperson is executive."
    },
    {
        "instruction": "What is the threshold for reporting suspicious transactions under FATF recommendations?",
        "response": "Under FATF recommendations, financial institutions are required to report suspicious transactions promptly to the Financial Intelligence Unit (FIU) regardless of the amount involved."
    },
    {
        "instruction": "What are the minimum capital adequacy standards under Basel III rules?",
        "response": "Basel III frameworks mandate a minimum Tier 1 capital ratio of 6% and a total Capital Adequacy Ratio (CAR) of 8% plus conservation buffers."
    }
] * 10  # Expanded for dummy training scale

# Fixed random seed split (80% Train, 20% Evaluation)
random.seed(42)
random.shuffle(sample_dataset)

split_idx = int(len(sample_dataset) * 0.8)
train_data = sample_dataset[:split_idx]
eval_data = sample_dataset[split_idx:]

# Save out to instruction_dataset.jsonl
with open("instruction_dataset.jsonl", "w", encoding="utf-8") as f:
    for item in sample_dataset:
        f.write(json.dumps(item) + "\n")

print(f"Dataset split completed: {len(train_data)} train samples, {len(eval_data)} evaluation samples.")


Dataset split completed: 40 train samples, 10 evaluation samples.


In [6]:
# import torch
# from datasets import Dataset
# from transformers import (
#     AutoModelForCausalLM,
#     AutoTokenizer,
#     BitsAndBytesConfig,
#     TrainingArguments
# )
# from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
# from trl import SFTTrainer

# MODEL_ID = "openai-community/gpt2-medium"

# # 1. LOAD TOKENIZER (With Explicit Padding Fix for GPT-2)
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# tokenizer.pad_token = tokenizer.eos_token

# # 2. BITSANDBYTES 4-BIT CONFIGURATION
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16
# )

# # Load Base Model
# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     quantization_config=bnb_config,
#     device_map="auto"
# )
# base_model = prepare_model_for_kbit_training(base_model)

# # 3. LORA ADAPTER CONFIGURATION (Adapter B - Balanced)
# peft_config = LoraConfig(
#     r=16,
#     lora_alpha=32,
#     target_modules=["c_attn"], # "c_attn" contains q, k, v projections combined inside standard GPT2 models
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM"
# )

# model = get_peft_model(base_model, peft_config)
# model.print_trainable_parameters()

# # 4. PREPARE DATASETS
# def formatting_prompts_func(example):
#     output_texts = []
#     for i in range(len(example['instruction'])):
#         text = f"### Instruction:\n{example['instruction'][i]}\n\n### Response:\n{example['response'][i]}"
#         output_texts.append(text)
#     return output_texts

# train_dataset = Dataset.from_list(train_data)
# eval_dataset = Dataset.from_list(eval_data)

# from trl import SFTConfig, SFTTrainer

# # 5. CONFIGURE SFT CONFIG (Replaces TrainingArguments)
# training_args = SFTConfig(
#     output_dir="./gpt2_compliance_adapter",
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=4,
#     eval_strategy="steps",
#     eval_steps=10,
#     logging_steps=5,
#     learning_rate=2e-4,
#     bf16=True,
#     max_steps=40,
#     save_strategy="steps",
#     save_steps=20,
#     weight_decay=0.01,
#     report_to="none",
#     max_length=512  # <-- FIXED: 'max_seq_length' is now 'max_length' inside SFTConfig
# )

# # 6. SFT TRAINER RUN
# trainer = SFTTrainer(
#     model=model,
#     train_dataset=train_dataset,
#     eval_dataset=eval_dataset,
#     peft_config=peft_config,
#     processing_class=tokenizer, # <-- FIXED: Use processing_class instead of tokenizer for newer TRL compatibility
#     formatting_func=formatting_prompts_func,
#     args=training_args          # <-- Pass the unified SFTConfig here
# )

# print("\nStarting Fine-Tuning Loop...")
# trainer.train()

# # Save final clean adapter weights
# trainer.model.save_pretrained("./final_compliance_adapter")
# print("Fine-tuning complete! Adapter configurations saved locally.")


In [9]:
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

MODEL_ID = "openai-community/gpt2-medium"

# 1. LOAD TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# 2. BITSANDBYTES 4-BIT CONFIGURATION
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load the Base Model (Keep it unwrapped here)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
base_model = prepare_model_for_kbit_training(base_model)

# 3. LORA ADAPTER CONFIGURATION (Adapter B - Balanced)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)


# =====================================================================
# STEP 4: PREPARE DATASETS (Explicit Text Column Pre-Mapping)
# =====================================================================
train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

# Map everything into a single, clean 'text' column that SFTTrainer looks for natively
def map_to_text_format(example):
    return {
        "text": f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['response']}"
    }

train_dataset = train_dataset.map(map_to_text_format)
eval_dataset = eval_dataset.map(map_to_text_format)

print("Sample dataset text structure ready:")
print(train_dataset[0]["text"])

# =====================================================================
# STEP 5: CONFIGURE SFT CONFIG (Updated with dataset_text_field)
# =====================================================================
training_args = SFTConfig(
    output_dir="./gpt2_compliance_adapter",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    eval_strategy="steps",
    eval_steps=10,
    logging_steps=5,
    learning_rate=2e-4,
    bf16=True,
    max_steps=40,
    save_strategy="steps",
    save_steps=20,
    weight_decay=0.01,
    report_to="none",
    max_length=512,
    dataset_text_field="text"  # <-- FIXED: Moved from SFTTrainer to SFTConfig
)

# =====================================================================
# STEP 6: SFT TRAINER RUN
# =====================================================================
trainer = SFTTrainer(
    model=base_model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args          # The trainer reads dataset_text_field from here now
)

print("\nStarting Fine-Tuning Loop...")
trainer.train()

# Save final clean adapter weights
trainer.model.save_pretrained("./final_compliance_adapter")
print("\n🎉 Success! Fine-tuning complete.")


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Sample dataset text structure ready:
### Instruction:
What is the maximum LTV ratio for gold loans as per RBI guidelines?

### Response:
Per RBI Master Direction on Loans and Advances, LTV for gold loans must not exceed 75% of the value of gold ornaments accepted as collateral.


Adding EOS to train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]


Starting Fine-Tuning Loop...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,3.874497,3.627057,3.558305,4470.000000,0.437795
20,3.500460,3.322772,3.582113,8940.000000,0.428691
30,3.225554,3.062833,3.511479,13410.000000,0.437640
40,3.092781,2.983297,3.463575,17880.000000,0.467344



🎉 Success! Fine-tuning complete.
